# Поиск маршрутов по сегментированной карте

Пайплайн: **изображение → модель → cost-грид → K маршрутов → визуализация**.

На вход нужна только картинка карты (+ чекпоинт модели).
`meta.json` рядом с `image.png` подхватывается сам (метры/минуты).
`use_gt=True` — опциональная отладка роутинга по готовой маске, без модели.

Сегментация кэшируется в `runs/routing_cache/`.


In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Не найден src/. Укажите ROOT вручную.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np

from src.routing import (
    DEFAULT_SPEEDS_MPS,
    animate_path_search,
    copy_speeds,
    compute_routes,
    format_route_description,
    prepare_segmentation,
    show_route_table,
    show_routes,
    show_segmentation,
)
from src.utils.config import load_experiment_config
from src.utils.device import get_device

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Параметры

Укажите путь к изображению и чекпоинт. Остальное подтянется само.


In [ ]:
# Единственный обязательный вход для карты — изображение:
IMAGE_PATH = ROOT / "dataset" / "kurakina_dacha_2017_omaps" / "image.png"
# IMAGE_PATH = ROOT / "dataset" / "ufa_komsomolsky_omaps" / "image.png"

CONFIG = "configs/quality_segformer.yaml"
CHECKPOINT = ROOT / "checkpoints" / "quality_segformer_b4" / "best.pt"

# Отладка роутинга без модели (берёт label.png рядом с картинкой):
USE_GT = not CHECKPOINT.exists()

CELL_SIZE = 4          # даунсемпл грида (больше → быстрее, грубее)
K_ROUTES = 3
PENALTY_FACTOR = 3.5   # сила штрафа вдоль уже найденных путей
PENALTY_WIDTH = 2      # ширина штраф-коридора (ячейки)
CONTOUR_PENALTY = 2.0  # рельеф: time *= (1 + penalty * доля_горизонталей); 0 = выкл.

assert IMAGE_PATH.exists(), IMAGE_PATH
print("image:", IMAGE_PATH)
print("checkpoint:", CHECKPOINT, "(exists)" if CHECKPOINT.exists() else "(нет → USE_GT)")
print("USE_GT:", USE_GT)


## 2. Загрузка карты и сегментация

In [ ]:
cfg = load_experiment_config(CONFIG)
device = get_device()
print("device:", device)

model = None
ckpt = None
if not USE_GT:
    from src.infer_fullmap import load_model_from_checkpoint
    assert CHECKPOINT.exists(), f"Нет чекпоинта: {CHECKPOINT}"
    model = load_model_from_checkpoint(CHECKPOINT, cfg, device)
    ckpt = CHECKPOINT

# Только IMAGE_PATH: meta.json ищется рядом сам; маска — из модели (или GT при USE_GT).
image, label, meta = prepare_segmentation(
    IMAGE_PATH,
    model=model,
    device=device,
    cfg=cfg,
    checkpoint=ckpt,
    use_gt=USE_GT,
    postprocess=True,
    use_cache=True,
)

print(f"image {image.shape[1]}×{image.shape[0]}, classes={len(np.unique(label))}")
print(f"resolution = {meta.resolution_m_per_px} м/px" if meta.has_scale else "масштаб неизвестен (время условное)")

from src.routing import render_segmentation_map

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(image)
axes[0].set_title("Исходная карта")
axes[0].axis("off")
axes[1].imshow(render_segmentation_map(label))
axes[1].set_title("Сегментация (рендер)")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 3. Точки старта и финиша

Координаты в **пикселях изображения** `(x, y)` — x вправо, y вниз.

Для выбора кликом можно поставить `%matplotlib widget` (нужен `ipympl`) и кликать по карте.

In [ ]:
h, w = image.shape[:2]

# стартовые точки — примерно через карту; поправьте под свой сценарий
START_XY = (w * 0.20, h * 0.30)
GOAL_XY = (w * 0.75, h * 0.70)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image)
ax.scatter([START_XY[0]], [START_XY[1]], c="lime", s=120, edgecolors="k", zorder=5, label="старт")
ax.scatter([GOAL_XY[0]], [GOAL_XY[1]], c="red", s=160, marker="*", edgecolors="k", zorder=5, label="финиш")
ax.legend()
ax.set_title("Старт / финиш")
ax.axis("off")
plt.show()

print(f"START={START_XY}, GOAL={GOAL_XY}")

## 4. Скорости по классам (опционально)

`DEFAULT_SPEEDS_MPS` — относительные м/с. `0` = непроходимо.  
Можно крутить и пересчитывать маршруты без повторной сегментации.

Горизонтали (`contours`) — overlay: класс местности берётся у соседей.  
Крутизна задаётся отдельно через `CONTOUR_PENALTY` (плотность горизонталей в ячейке).

In [ ]:
speeds = copy_speeds(DEFAULT_SPEEDS_MPS)

# примеры тюнинга:
# speeds["green_hard"] = 0.05   # почти непроходимо
# speeds["marsh"] = 0.08
# speeds["road"] = 2.5          # ещё быстрее по дороге

for name, v in sorted(speeds.items(), key=lambda kv: -kv[1]):
    print(f"{name:20s} {v:6.3f} м/с")

## 5. Поиск K маршрутов

In [ ]:
cost, routes = compute_routes(
    label,
    meta,
    START_XY,
    GOAL_XY,
    speeds=speeds,
    cell_size=CELL_SIZE,
    k=K_ROUTES,
    penalty_factor=PENALTY_FACTOR,
    penalty_width=PENALTY_WIDTH,
    contour_penalty=CONTOUR_PENALTY,
)

print(f"cost-grid {cost.shape[1]}×{cost.shape[0]} (cell={CELL_SIZE}px = {cost.cell_m:.2f} м)")
if cost.contour_density is not None:
    d = cost.contour_density
    print(
        f"contour penalty={cost.contour_penalty}: "
        f"density mean={d.mean():.3f}, max={d.max():.3f}"
    )
if not routes:
    print("Маршрут не найден (точки на разных «островах»?).")
else:
    for r in routes:
        print(format_route_description(r))
        print("─" * 40)

## 6. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
show_routes(
    image, routes,
    start_xy=START_XY, goal_xy=GOAL_XY,
    background="image",
    ax=axes[0],
    title="Исходная карта",
)
show_routes(
    image, routes,
    start_xy=START_XY, goal_xy=GOAL_XY,
    label=label,
    background="segmentation",  # карта, отрендеренная с нуля по классам
    ax=axes[1],
    title="Маршруты на сегментации",
)
plt.tight_layout()
plt.show()


## 6.5. Анимация A* (волна поиска)

Визуализация закрытых ячеек A* по порядку посещения, затем отрисовка найденного пути.
GIF сохраняется в `runs/routing_vis/` и сразу показывается в ячейке.

In [ ]:
# Анимация распространения волны A* + построение кратчайшего пути
# (использует тот же cost-грид и точки START/GOAL, что и секция 5)

trace, anim_path = animate_path_search(
    image,
    cost,
    START_XY,
    GOAL_XY,
    out_path=ROOT / "runs" / "routing_vis" / "astar_wave.gif",
    format="gif",       # или "mp4"
    fps=22,
    n_wave_frames=100,  # кадров волны
    n_path_frames=50,   # кадров отрисовки пути
    hold_frames=30,     # пауза на финальном кадре
    max_side=1280,      # даунскейл длинной стороны (None = полный размер)
    display=True,
)

print(f"сохранено: {anim_path}")
print(f"закрыто ячеек: {trace.n_closed}")
if trace.route is None:
    print("Путь не найден.")
else:
    print(
        f"путь: {trace.route.distance_m:.0f} м, "
        f"{trace.route.time_min:.1f} мин, "
        f"{len(trace.route.cells)} ячеек"
    )

## 7. Описание местности по маршруту

In [ ]:
ROUTE_IDX = 0  # 0 = самый быстрый

if routes:
    df = show_route_table(routes[ROUTE_IDX])
    display(df)
else:
    print("Нет маршрутов.")

## 8. Интерактив (опционально)

Раскомментируйте, если установлены `ipywidgets` (+ `ipympl` для кликов).

In [ ]:
# # --- клик по карте для выбора точек ---
# # %pip install ipympl ipywidgets
# # %matplotlib widget
#
# clicks = []
# fig, ax = plt.subplots(figsize=(9, 9))
# ax.imshow(image)
# ax.set_title("Клик 1 = старт, клик 2 = финиш")
# ax.axis("off")
#
# def on_click(event):
#     if event.inaxes != ax or event.xdata is None:
#         return
#     clicks.append((float(event.xdata), float(event.ydata)))
#     ax.scatter([event.xdata], [event.ydata], c="cyan", s=80, edgecolors="k")
#     fig.canvas.draw_idle()
#     if len(clicks) >= 2:
#         global START_XY, GOAL_XY
#         START_XY, GOAL_XY = clicks[-2], clicks[-1]
#         print("START", START_XY, "GOAL", GOAL_XY)
#
# fig.canvas.mpl_connect("button_press_event", on_click)

In [ ]:
# # --- слайдеры скоростей + пересчёт ---
# from ipywidgets import FloatSlider, IntSlider, Button, VBox, HBox, Output
# from IPython.display import display, clear_output
#
# out = Output()
# sliders = {
#     name: FloatSlider(value=v, min=0.0, max=3.0, step=0.05, description=name[:12])
#     for name, v in DEFAULT_SPEEDS_MPS.items()
#     if name not in ("background",)
# }
# k_slider = IntSlider(value=K_ROUTES, min=1, max=5, description="K")
# cell_slider = IntSlider(value=CELL_SIZE, min=1, max=16, description="cell")
# btn = Button(description="Пересчитать", button_style="primary")
#
# def recalc(_=None):
#     with out:
#         clear_output(wait=True)
#         sp = {n: s.value for n, s in sliders.items()}
#         sp.setdefault("background", DEFAULT_SPEEDS_MPS["background"])
#         _, rts = compute_routes(
#             label, meta, START_XY, GOAL_XY,
#             speeds=sp, cell_size=cell_slider.value, k=k_slider.value,
#             penalty_factor=PENALTY_FACTOR, penalty_width=PENALTY_WIDTH,
#         )
#         fig, ax = plt.subplots(figsize=(10, 8))
#         show_routes(image, rts, start_xy=START_XY, goal_xy=GOAL_XY, ax=ax)
#         plt.show()
#         if rts:
#             display(show_route_table(rts[0]))
#
# btn.on_click(recalc)
# display(VBox([HBox([k_slider, cell_slider, btn]), *sliders.values(), out]))
# recalc()